# Ingest Sprints JSON Files

This notebook demonstrates the ingestion of sprints JSON files (one per season) into the bronze layer.

## What this notebook covers
* Load the sprints source files (one JSON per season)
* Define and enforce schema
* Add ingestion metadata columns: source_file and ingestion_timestamp
* Write the transformed data to the bronze Delta table
* Validate the loaded sprints

This uses centralized configuration and helper functions for consistency.

In [0]:
%run ../00-Common/01.Formula1_Configuration_Setup

In [0]:
# Check configured catalog
catalog_name

In [0]:
%run ../00-Common/02.Bronze-Helper

In [0]:
# Check landing folder path
landing_folder_path

In [0]:
# Define source folder and target table
source_folder = f"{landing_folder_path}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

In [0]:
# Preview source folder path
source_folder

In [0]:
# Define the schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

sprints_schema = StructType([
    StructField("constructorId", StringType()),
    StructField("date",          StringType()),
    StructField("driverId",      StringType()),
    StructField("grid",          IntegerType()),
    StructField("laps",          IntegerType()),
    StructField("number",        IntegerType()),
    StructField("points",        DoubleType()),
    StructField("position",      IntegerType()),
    StructField("positionText",  StringType()),
    StructField("raceName",      StringType()),
    StructField("round",         IntegerType()),
    StructField("season",        IntegerType()),
    StructField("status",        StringType()),
    StructField("url",           StringType())
])

In [0]:
# Read all sprints JSON files from the directory into a DataFrame
sprints_df = (
    spark.read
    .format("json")
    .schema(sprints_schema)
    .option("multiLine", True)
    .option("mode", "FAILFAST")
    .load(source_folder)
)

display(sprints_df)

In [0]:
# Add ingestion metadata columns
sprints_final_df = add_ingestion_metadata(sprints_df)
display(sprints_final_df)

In [0]:
# Write sprints data to the bronze Delta table
(
    sprints_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

In [0]:
# Preview target table name
table_name

In [0]:
%sql
-- Query the bronze sprints table
SELECT * FROM formula1.bronze.sprints LIMIT 10

In [0]:
# Display the bronze sprints table
display(spark.table(table_name))